In [ ]:
import os
import sys
import pandas as pd
from typing import Dict

In [ ]:
curr_dir = os.getcwd()
parent_dir = os.path.dirname(curr_dir)
proj_dir = os.path.dirname(parent_dir)
sys.path.append(proj_dir)

In [ ]:
from baseline.turbulence_benchmark.utility.turbulence_log_functions import TurbulenceLogHelper

In [ ]:
def generate_results_table(res_dir: str):

    """
    This method assumes that all logs are filled. 
    """

    csv_logs = [f for f in os.listdir(res_dir) if 
        os.path.isfile(os.path.join(res_dir, f)) and 
        f.endswith(".csv") and 
        "turbulence" in f.lower()
        ]
    
    print(csv_logs)
    
    if len(csv_logs) != 3:
        raise IndexError("More than 3 Turbulence Benchmark logs in results directory.")
    
    no_mutation_log_name = [f for f in csv_logs if "no_mutation" in f][-1]
    random_log_name = [f for f in csv_logs if "random" in f][-1]
    sequential_log_name = [f for f in csv_logs if "sequential" in f][-1]        
    
    no_mutation_log = pd.read_csv(os.path.join(res_dir, no_mutation_log_name))
    random_log = pd.read_csv(os.path.join(res_dir, random_log_name))
    sequential_log = pd.read_csv(os.path.join(res_dir, sequential_log_name))

    helper = TurbulenceLogHelper()

    res_df = pd.DataFrame(
        index = ["Turbulence", "MuCoCo Random", "MuCoCo Sequential", "MuCoCo Aggregate"],
        columns = ["No. of questions", "No. of tasks"]
    )

    res_df.loc[:, "No. of questions"] = helper.total_questions
    res_df.loc[:, "No. of tasks"] = helper.total_tasks

    print("no_mut")
    turbulence_dict = helper.obtain_turbulence_code_inconsistency_score(no_mutation_log)
    # random_inconsistency_score, random_inconsistency_percentage = helper.obtain_mucoco_code_inconsistency_score(log1= no_mutation_log, log2=random_log)
    # seq_inconsistency_score, seq_inconsistency_percentage = helper.obtain_mucoco_code_inconsistency_score(log1= no_mutation_log, log2=sequential_log)
    print('random')
    random_dict = helper.obtain_turbulence_code_inconsistency_score(log=random_log)
    print('seq')
    sequential_dict = helper.obtain_turbulence_code_inconsistency_score(log=sequential_log)

    turbulence_qn_inconsistency_dict = helper.obtain_question_inconsistency_count(log = no_mutation_log)
    random_qn_inconsistency_dict = helper.obtain_question_inconsistency_count(log = random_log)
    sequential_qn_inconsistency_dict = helper.obtain_question_inconsistency_count(log = sequential_log)

    # Updating inconsistency scores
    res_df.loc["Turbulence", "Code Inconsistency Score"] = f"{turbulence_dict['inconsistency_count']}/{turbulence_dict['total_comparisons']}"
    res_df.loc["MuCoCo Random", "Code Inconsistency Score"] = f"{random_dict['inconsistency_count']}/{random_dict['total_comparisons']}"
    res_df.loc["MuCoCo Sequential", "Code Inconsistency Score"] = f"{sequential_dict['inconsistency_count']}/{sequential_dict['total_comparisons']}"
    aggregate_inconsistency_score = (random_dict['inconsistency_count'] + sequential_dict['inconsistency_count'])//2
    res_df.loc["MuCoCo Aggregate", "Code Inconsistency Score"] = f"{aggregate_inconsistency_score}/{sequential_dict['total_comparisons']}"


    # Updating inconsistency percentages
    res_df.loc["Turbulence", "Code Inconsistency %"] = f"{round(turbulence_dict['inconsistency_count']*100 / turbulence_dict['total_comparisons'], 2)}"
    res_df.loc["MuCoCo Random", "Code Inconsistency %"] = f"{round(random_dict['inconsistency_count']*100 / random_dict['total_comparisons'],2 )}"
    res_df.loc["MuCoCo Sequential", "Code Inconsistency %"] = f"{round(sequential_dict['inconsistency_count']*100 / sequential_dict['total_comparisons'], 2)}"
    res_df.loc["MuCoCo Aggregate", "Code Inconsistency %"] = f"{round(aggregate_inconsistency_score*100/sequential_dict['total_comparisons'], 2)}"


    # Updating question inconsistency
    res_df.loc["Turbulence", "Question Inconsistency"] = f"{turbulence_qn_inconsistency_dict['inconsistent_qn_count']}/{turbulence_qn_inconsistency_dict['total_questions']}"
    res_df.loc["MuCoCo Random", "Question Inconsistency"] = f"{random_qn_inconsistency_dict['inconsistent_qn_count']}/{random_qn_inconsistency_dict['total_questions']}"
    res_df.loc["MuCoCo Sequential", "Question Inconsistency"] = f"{sequential_qn_inconsistency_dict['inconsistent_qn_count']}/{sequential_qn_inconsistency_dict['total_questions']}"
    aggregate_qn_inconsistency = (random_qn_inconsistency_dict['inconsistent_qn_count'] + sequential_qn_inconsistency_dict['inconsistent_qn_count'])//2
    res_df.loc["MuCoCo Aggregate", "Question Inconsistency"] = f"{aggregate_qn_inconsistency}/{sequential_qn_inconsistency_dict['total_questions']}"

    # Updating question inconsistency
    res_df.loc["Turbulence", "Question Inconsistency %"] = f"{round(turbulence_qn_inconsistency_dict['inconsistent_qn_count']*100 /turbulence_qn_inconsistency_dict['total_questions'], 2)}"
    res_df.loc["MuCoCo Random", "Question Inconsistency %"] = f"{round(random_qn_inconsistency_dict['inconsistent_qn_count']*100 /random_qn_inconsistency_dict['total_questions'], 2)}"
    res_df.loc["MuCoCo Sequential", "Question Inconsistency %"] = f"{round(sequential_qn_inconsistency_dict['inconsistent_qn_count']*100 /sequential_qn_inconsistency_dict['total_questions'], 2)}"
    res_df.loc["MuCoCo Aggregate", "Question Inconsistency %"] = f"{round(aggregate_qn_inconsistency*100/sequential_qn_inconsistency_dict['total_questions'], 2)}"

    return res_df

res_dir = proj_dir + "/Users/jin/Downloads/Test_results_26:09:2025/code_generation/gpt-4o"
res_dir = "/Users/jin/Downloads/Test_results_26:09:2025/code_generation/gpt-4o"

# res_dir = "/Users/jin/Downloads/gpt-4o"
generate_results_table(res_dir=res_dir)

In [ ]:
def generate_results_table(res_dir: str):

    """
    This method assumes that all logs are filled. 
    """

    csv_logs = [f for f in os.listdir(res_dir) if 
        os.path.isfile(os.path.join(res_dir, f)) and 
        f.endswith(".csv") and 
        "turbulence" in f.lower()
        ]
    
    print(csv_logs)
    
    no_mutation_log_name = [f for f in csv_logs if "no_mutation" in f][-1]
    random_log_name = [f for f in csv_logs if "random" in f][-1]
    sequential_log_name = [f for f in csv_logs if "sequential" in f][-1]      
    for2while_log_name = [f for f in csv_logs if "for2while" in f][-1]      
    for2enumerate_log_name = [f for f in csv_logs if "for2enumerate" in f][-1]
    literal_format_log_name = [f for f in csv_logs if "literal_format" in f][-1]
    boolean_literal_log_name = [f for f in csv_logs if "boolean_literal" in f][-1]
    commutative_reorder_log_name = [f for f in csv_logs if "commutative_reorder" in f][-1]
    demorgan_log_name = [f for f in csv_logs if "demorgan" in f][-1]
    const_unfold_log_name = [f for f in csv_logs if "constant_unfold" in f and not any(suffix in f for suffix in ["constant_unfold_add", "constant_unfold_mult"])][-1]
    const_unfold_multi_log_name = [f for f in csv_logs if "constant_unfold_add" in f][-1]
    const_unfold_add_log_name = [f for f in csv_logs if "constant_unfold_mult" in f][-1]

    log_names = [no_mutation_log_name, random_log_name, sequential_log_name, for2while_log_name, for2enumerate_log_name, literal_format_log_name, boolean_literal_log_name, commutative_reorder_log_name, demorgan_log_name, const_unfold_log_name, const_unfold_add_log_name, const_unfold_multi_log_name]

    helper = TurbulenceLogHelper()

    res_df = pd.DataFrame(
        index = [],
        columns = ["No. of questions", "No. of tasks"]
    )

    for log_name in log_names:
        
        log = pd.read_csv(os.path.join(res_dir, log_name))

        log_dict: Dict = helper.obtain_turbulence_code_inconsistency_score(log)
        log_qn_inconsistency_dict = helper.obtain_question_inconsistency_count(log = log)

        res_df.loc[log_name.capitalize(), "No. of questions"] = f"{log_qn_inconsistency_dict['total_questions']}"
        res_df.loc[log_name.capitalize(), "No. of tasks"] =f"{log_qn_inconsistency_dict['num_questions']}"


        res_df.loc[log_name.capitalize(), "Model Accuracy"] = f"{round(log_dict['correct_instances']*100 / (log_dict['incorrect_instances'] + log_dict['correct_instances']), 2)}"
        res_df.loc[log_name.capitalize(),"Code Inconsistency Score"] = f"{log_dict['inconsistency_count']}/{log_dict['total_comparisons']}"
        res_df.loc[log_name.capitalize(), "Code Inconsistency %"] = f"{round(log_dict['inconsistency_count']*100 / log_dict['total_comparisons'], 2)}"


        res_df.loc[log_name.capitalize(), "Question Inconsistency"] = f"{log_qn_inconsistency_dict['inconsistent_qn_count']}/{log_qn_inconsistency_dict['total_questions']}"
        res_df.loc[log_name.capitalize(), "Question Inconsistency %"] = f"{round(log_qn_inconsistency_dict['inconsistent_qn_count']*100 /log_qn_inconsistency_dict['total_questions'], 2)}"

    return res_df

res_dir =  "/Users/jin/Downloads/Test_results_26:09:2025/input_prediction/gpt-4o"

# res_dir = "/Users/jin/Downloads/gpt-4o"
generate_results_table(res_dir=res_dir)

In [ ]:
res_dir =  "/Users/jin/Downloads/Test_results_26:09:2025/output_prediction/gpt-4o"

# res_dir = "/Users/jin/Downloads/gpt-4o"
generate_results_table(res_dir=res_dir)